# CAJAL Computational Mini-Project 16

## Level 2 — Exploring the gene regulatory networks underlying morphogen-patterned neuronal identity

---

You have been given a single-cell RNA-seq dataset from a large-scale combinatorial morphogen perturbation screen. The dataset was generated using Parse single cell technology and there are 2 plates and 192 perturbations in total. You have already analysed and annotated the screen, and you have seen how each morphogen combination shifts the regional and neuronal type composition of the culture.

Composition tells you *which* cell types appear, but not *how* they are specified. Morphogens do not act on cell identity directly — they switch on transcription factors, and each transcription factor in turn drives a programme of downstream target genes. A transcription factor together with the targets it controls is called a **regulon**, and the collection of regulons active in a cell is its **gene regulatory network** (GRN).

In this project you will infer regulons directly from the expression data using **SCENIC**, score how active each regulon is in each cell, and then ask the central question: **which regulons does each morphogen switch on, and where in the nervous system are those regulons active?**

Your goal is to recreate **figure 3E** of the [source paper](https://www.science.org/doi/10.1126/science.adn6121) — a network with the morphogens at the centre, linked to the regulons they activate, with each regulon coloured by the region in which it is most active. If your analysis works, you should be able to rediscover what the authors reported:

> *RA-associated regulons aligned with hindbrain fates, including well-known drivers of posterior patterning such as HOXB4, RARB, FOXO3, and MEIS1. FGF8-associated regulons had the propensity to promote midbrain fates, while BMP4 and CHIR-associated regulons exhibited strong links to the PNS.*

**Important:** SCENIC is computationally expensive. The authors ran it on the full dataset and repeated it over 20 independent subsamples to build a consensus network. You have a single node and an afternoon, so you will run it once, on a small balanced sample of cells. Your figure will be a rougher version of the paper's, but the main biological story should still come through. Precomputed results are provided at each stage in `data/`, so a slow step never blocks you from finishing the project.

**Note:** Use the [pySCENIC documentation](https://pyscenic.readthedocs.io/en/latest/), the [pySCENIC tutorial](https://pyscenic.readthedocs.io/en/latest/tutorial.html) and the [example notebooks](https://github.com/aertslab/pySCENIC/tree/master/notebooks) in the pySCENIC repository as a starting point for your analysis. The two method papers are also worth reading: [Aibar et al. (2017)](https://doi.org/10.1038/nmeth.4463) introduces SCENIC, and [Van de Sande et al. (2020)](https://doi.org/10.1038/s41596-020-0336-2) describes the workflow you are about to run in detail.

**Before you start:** this notebook must be run with the `pySCENIC scanpy 1.10.4 / anndata 0.11.4 (SIF)` kernel — pySCENIC and its dependencies are not installed in the default environment. Check the kernel selector in the top right.

In [ ]:
import os
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"

import glob
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx

# Required when obs/var contain pandas StringArray columns.
pd.options.future.infer_string = False

sc.settings.verbosity = 1
sc.settings.set_figure_params(dpi=100, frameon=False, figsize=(5, 4))
sns.set_style("whitegrid")

%matplotlib inline

In [ ]:
# ---- file paths --------------------------------------------------------------
DATA = "data/ngn2_small.h5ad"   # ~9,900 neurons, balanced across cell types
PRE = "data"                    # precomputed results, if a step runs too slowly

# The complete dataset (~188,000 cells, slow to load) if you ever want it
FULL_DATA = ("/shared/projects/tp_2630_ubordeaux_neuromics_184418/projects/C16/shared/ngn2_scrna_processed_scenic.h5ad")

# ---- cisTarget motif databases (shared, read-only) --------------------------
RES = ("/shared/projects/tp_2630_ubordeaux_neuromics_184418/projects/C16/shared/resources/cistarget")
TF_NAMES = f"{RES}/allTFs_hg38.txt"                              # human transcription factors
RANKINGS = sorted(glob.glob(f"{RES}/*.genes_vs_motifs.rankings.feather"))  # motif rankings
MOTIF_TBL = f"{RES}/motifs-v10nr_clust-nr.hgnc-m0.001-o0.0.tbl"   # motif -> TF annotations

# ---- the six morphogens varied in the screen --------------------------------
# The obs columns "M_XAV", "M_CHIR", ... hold the concentration each cell received.
MORPHOGENS = ["XAV", "CHIR", "RA", "FGF8", "BMP4", "SHH"]

# ---- colours (the same palette as the composition notebook) -----------------
REGION_ORDER = ["Forebrain", "Midbrain", "Hindbrain", "Spinal cord",
                "SYM", "ENS", "TG", "DRG"]
REGION_COLORS = {"Forebrain": "#fe9b00", "Midbrain": "#f4c40f",
                 "Hindbrain": "#d8443c", "Spinal cord": "#9b3441",
                 "SYM": "#268a8a", "ENS": "#226a99",
                 "TG": "#383b81", "DRG": "#92c051"}

## Part 1: Data Loading and Subsampling

SCENIC's runtime scales with the number of cells, and the first step — inferring co-expression between every transcription factor and every gene — is by far the most expensive. Running it on all ~188,000 cells would take many hours. Instead you will work with `ngn2_small.h5ad`, a pre-prepared file of ~9,900 neurons that loads in seconds, and subsample it further.

How you subsample matters. If you draw cells at random, abundant cell types dominate and the regulons specific to rare populations are never detected. A **balanced** sample — the same number of cells from every cell type — keeps every population represented, which is what you need if the final figure is to span all eight regions.

Load the data and take a balanced sample of cells across the `cell_type` annotation. Something in the range of 15–30 cells per type is sensible: more cells give you more regulons, and therefore a denser and more reliable network, but a slower SCENIC run. Print the size of your sample and check the `Region` composition to confirm that no region has been lost. You should also filter out genes detected in only a handful of cells, since they carry no usable co-expression signal and only inflate the runtime.

Take a moment to inspect the object before running anything expensive. What is in `.X` — raw counts, or normalised and log-transformed values? Which `obs` columns encode the morphogen concentrations, the region and the cell type? GRNBoost2 (the first SCENIC step) expects a *dense* cells × genes matrix with gene symbols as column names, not a sparse AnnData object, so you will need to convert it. Build that expression table now — you will reuse it throughout Part 2.

## Part 2: Inferring Regulons with SCENIC

SCENIC turns an expression matrix into a set of regulons and a per-cell activity score for each of them. It does this in three steps, and it is worth understanding why each one exists before you write any code.

1. **GRNBoost2** asks, for every transcription factor, which genes have expression that tracks it across cells. It does this by training a gradient-boosting model to predict each gene's expression from the expression of all transcription factors, and reading off the importance of each TF. The output is a long list of weighted TF → gene links, called *adjacencies*. These are purely correlative: a TF and a gene may co-vary because one drives the other, or because both respond to something else entirely.

2. **cisTarget** removes the false positives. A genuine target of a transcription factor should carry that factor's DNA binding motif in its regulatory region. cisTarget groups the adjacencies into candidate modules, then checks each module for enrichment of the TF's motif, using precomputed genome-wide motif rankings. Modules that survive become **regulons**: a transcription factor plus the targets it plausibly binds. This is what separates SCENIC from a plain co-expression network.

3. **AUCell** scores activity. For each cell it ranks all genes by expression and asks how enriched a regulon's targets are near the top of that ranking, giving an area-under-the-recovery-curve value per cell per regulon. The result is a cells × regulons matrix that you can treat much like an expression matrix — but where each column is a *regulatory programme* rather than a single gene.

The functions you need are spread across three packages, all installed in this kernel: `arboreto` (GRNBoost2), `ctxcore` (the motif ranking databases) and `pyscenic` (module construction, pruning, AUCell). Work out the calls from the [pySCENIC documentation](https://pyscenic.readthedocs.io/en/latest/) and the [tutorial](https://pyscenic.readthedocs.io/en/latest/tutorial.html) — note that the tutorial demonstrates both a command-line interface and a Python API, and you want the Python API here. The three resource files you will need are already assigned to `TF_NAMES`, `RANKINGS` and `MOTIF_TBL` in the setup cell above.

### 2.1. Co-expression modules

Run GRNBoost2 on your sampled expression matrix, using the human transcription factor list as the set of candidate regulators. Restrict that list to transcription factors that are actually present in your matrix — asking for a regulator that was filtered out will fail. Set a random seed so your network is reproducible.

GRNBoost2 parallelises over a Dask cluster. You do not need to create one yourself: if you do not pass a client, a local cluster is started for you, which is the right choice here. (Be aware that this convenience is a trap outside a notebook — in a plain `.py` script, Dask's worker processes re-import the script and re-run it recursively unless everything is guarded by `if __name__ == "__main__":`.)

Inspect the adjacencies once the run finishes. How many TF → gene links were returned, and how are the weights distributed?

### 2.2. Motif enrichment and pruning

Now convert the adjacencies into modules, prune them against the motif databases, and convert the surviving modules into regulons. Two of the ranking databases are provided — one scoring a window of 500 bp upstream to 100 bp downstream of the transcription start site, the other a much wider ±10 kb window — and pySCENIC will use both, keeping a target if it is supported in either.

This is the slowest cell in the notebook: expect anything from several minutes to considerably longer, since the machine is shared. Pass a sensible number of workers. If it has not finished in a time you are willing to wait, interrupt it and move on using the precomputed results described below — you will lose nothing but the satisfaction.

Once you have regulons, look at what you have: how many are there, how many target genes does a typical one control, and are the transcription factors named among them ones you recognise as neuronal?

### 2.3. Scoring regulon activity per cell

Score every cell in your sample for every regulon with AUCell. The result should be a cells × regulons matrix. Keep alongside it a `meta` table of the matching `obs` rows — the region and morphogen concentrations for exactly those cells, in exactly that order — because every remaining part of this notebook joins regulon activity to cell metadata, and a silent index mismatch here will quietly corrupt everything downstream.

**If a step above was too slow:** precomputed results are in `data/`. `auc_matrix.csv` is a cells × regulons AUCell matrix, `cell_metadata.csv` the matching per-cell metadata, and `regulon_sizes.csv` the number of target genes per regulon. Note that these were computed on a *different* random sample of cells than yours, so you must take both the matrix and the metadata from the precomputed files together — never mix your own sample with the provided one. It is worth structuring your code so that a single switch at the top decides whether to run SCENIC or load these files.

### 2.4. Regulon activity across regions

Before looking at morphogens, check that the regulons you inferred carry regional information at all. Average each regulon's activity within each region, giving a regulons × regions table. Raw activity scores are not comparable between regulons — a regulon with many targets tends to score higher everywhere — so z-score each regulon across regions. This tells you *where* a regulon is relatively high rather than how high it is.

Visualise the most region-specific regulons as a heatmap. Do the transcription factors driving each region make biological sense to you? A precomputed version of this table is in `data/regulon_mean_activity_by_region.csv` if you need it.

## Part 3: Linking Morphogens to Regulons

This is the heart of figure 3E. You have, for each cell, the concentration of each of the six morphogens it received, and the activity of each regulon. You want to know which morphogen drives which regulon.

The elegant trick the authors used is to run GRNBoost2 a *second* time, but with the roles changed: the **morphogen concentrations become the regulators**, and the **regulon activities become the targets**. The algorithm does not care that its "regulators" are now experimental doses rather than transcription factors — it simply asks which regulators predict each target, and returns weighted links. A strong morphogen → regulon link means that regulon's activity is well predicted by that morphogen's dose.

Assemble a single matrix with one row per cell whose columns are the regulon activities plus the six morphogen concentrations, drop any morphogen that does not vary in your sample, and run GRNBoost2 with the morphogen columns as the regulator list.

Two things to watch for when you turn the result into a tidy table of links:

* Building modules from these adjacencies emits **several modules per morphogen**, so the same (morphogen, regulon) pair appears repeatedly with an identical weight. Deduplicate, or your final figure will be filled with the same handful of links drawn over and over.
* Morphogen → morphogen links will appear. Drop them; only morphogen → regulon links belong in the figure.

Finally, filter out the weak links — the paper does this too — by keeping only each morphogen's strongest handful. Around eight links per morphogen gives a readable figure; raise it for a busier network, lower it for a cleaner one. A precomputed table is in `data/morphogen_regulon_links.tsv` (use the rows where `link_type` is `individual`).

## Part 4: Recreating Figure 3E

You now have everything the figure needs. In the paper's panel E, each **morphogen** is a large oval at the centre of the network, each **regulon** is a circle connected to the morphogens that drive it, the **colour** of a regulon encodes the region in which it is most active, its **size** encodes how many target genes it controls, and the **shade of each edge** encodes the strength of the link.

Work out each regulon's dominant region from the mean-activity table you built in Part 2.4, and its size from the number of genes in the regulon (or from `data/regulon_sizes.csv`). Then build the graph with `networkx` and draw it. A force-directed layout such as `nx.spring_layout` will place regulons shared between morphogens between them, which is exactly the structure you want to reveal — tune the repulsion and iterations until the labels are legible.

Use `REGION_COLORS` from the setup cell so that your figure is consistent with the plots from the previous notebook, and include a region legend.

## Part 5: Biological Interpretation

Now comes the biological interpretation. Look at the network you have generated and work out what it says about how morphogens pattern neuronal identity. Here are some questions to get you started:

* For each morphogen, which region do its linked regulons mostly belong to? Summarise this as a table rather than reading it off the figure by eye.
* The paper reports that RA-associated regulons align with **hindbrain** fates, naming HOXB4, RARB, FOXO3 and MEIS1. Which of these did you recover?
* BMP4- and CHIR-associated regulons are reported to link strongly to the **PNS**. Do you see this, and do PHOX2B or the SMAD factors appear among them?
* Do FGF8-associated regulons promote **midbrain** fates, as reported?
* Which morphogens, if any, produced links you did not expect? Is there a technical explanation — sample size, dose range, a morphogen that barely varies?

A link in the network says a morphogen predicts a regulon's activity, but not how. Morphogens act as **dose-dependent** signals, so a genuine relationship should show a graded response: as the concentration rises, the regulon's activity should rise (or fall) with it, rather than jumping arbitrarily between doses.

Pick one of the RA-linked regulons and plot its mean activity against the RA concentration. The doses span orders of magnitude and include zero, so a linear x-axis will crush most of the points together — a symmetric-log scale handles this. Then do the same for a morphogen whose biology you found surprising. Does the relationship look graded and monotonic, or noisy?

## Part 6: Making Figures

As in the previous notebooks, the plots you have made so far were for you — quick, exploratory, and readable only with the notebook beside them. Turn your network and one supporting panel into publication-quality figures.

A network figure is unusually easy to render badly. Check that node labels do not overlap or spill outside the axes, that the regulon colours remain distinguishable when printed, that the region legend is complete, and that a reader can tell what node size and edge shade encode without reading your code — which means either a caption or a size legend. Export in a vector format so that the labels stay sharp at any zoom.

## Part 7: Save Results and Summary

Save the regulon activity matrix, the morphogen → regulon links and your figures, so that your analysis can be reproduced without rerunning SCENIC.

Finally, write up what you found. Your summary should cover:

* Your version of figure 3E, and a sentence on which morphogen links to which region.
* Which of the paper's named drivers you recovered — HOXB4, RARB, FOXO3 and MEIS1 for RA; PHOX2B, SMAD1, SMAD9 and MSX1 for BMP4 — and which you did not.
* At least one limitation of running SCENIC on a sample of ~1,500 cells rather than the full dataset, and how the authors' use of 20 independent subsamples addresses it.
* One thing this analysis cannot tell you: the links are statistical associations between dose and regulon activity, not demonstrations that the morphogen acts through that transcription factor. What experiment would settle it?